In [ ]:
%%bash
cat << 'EOF' > /content/config.sh
#!/bin/bash

export ROOTDIR="/content/exterieur"
export VIDEOSOURCE="gsplat/input/IMG_4797.MOV"
export IMAGESET="gsplat/input/perfume/video"
export INPUT_MODE="video"

#export NUM_FRAMES=150
export FPS=3

#branche dev
export GIT_BRANCH="dev"
export BASENAME="exterieur"

export PROFILE="gpu/quality"
#export PROFILE="gpu/balanced"
#export PROFILE="cpu/fast"
EOF

In [ ]:
!cat /content/config.sh

#!/bin/bash

export ROOTDIR="/content/exterieur"
export VIDEOSOURCE="gsplat/input/IMG_4797.MOV"
export IMAGESET="gsplat/input/perfume/video"
export INPUT_MODE="video"

#export NUM_FRAMES=150
export FPS=3

#branche dev
export GIT_BRANCH="dev"
export BASENAME="exterieur"

export PROFILE="gpu/quality"
#export PROFILE="gpu/balanced"
#export PROFILE="cpu/fast"


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!wget -q https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh
!bash Miniforge3-Linux-x86_64.sh -b -p /usr/local/miniforge

# activer conda pour cette session notebook
import os
os.environ["PATH"] = "/usr/local/miniforge/bin:" + os.environ["PATH"]

!conda --version

PREFIX=/usr/local/miniforge
Unpacking bootstrapper...
Unpacking payload...
Extracting ca-certificates-2026.2.25-hbd8a1cb_0.conda
Extracting libgomp-15.2.0-he0feb66_18.conda
Extracting libzlib-1.3.2-h25fd6f3_2.conda
Extracting nlohmann_json-abi-3.12.0-h0f90c79_1.conda
Extracting pybind11-abi-11-hc364b38_1.conda
Extracting python_abi-3.13-8_cp313.conda
Extracting tzdata-2025c-hc9c84f9_1.conda
Extracting _openmp_mutex-4.5-20_gnu.conda
Extracting zstd-1.5.7-hb78ec9c_6.conda
Extracting ld_impl_linux-64-2.45.1-default_hbd61a6d_101.conda
Extracting libgcc-15.2.0-he0feb66_18.conda
Extracting bzip2-1.0.8-hda65f42_9.conda
Extracting c-ares-1.34.6-hb03c661_0.conda
Extracting keyutils-1.6.3-hb9d3cd8_0.conda
Extracting libexpat-2.7.4-hecca717_0.conda
Extracting libffi-3.5.2-h3435931_0.conda
Extracting libgcc-ng-15.2.0-h69a702a_18.conda
Extracting libiconv-1.18-h3b78370_2.conda
Extracting liblzma-5.8.2-hb03c661_0.conda
Extracting libmpdec-4.0.0-hb03c661_1.conda
Extracting libstdcxx-15.2.0-h934c35e_1

In [ ]:
!RUN=0; \
[ "$RUN" -eq 0 ] && echo "skipping this stage" || \
(wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh && \
chmod +x Miniconda3-latest-Linux-x86_64.sh  && \
bash Miniconda3-latest-Linux-x86_64.sh -b -p /usr/local/miniconda  && \
/usr/local/miniconda/bin/conda init bash)

skipping this stage


In [ ]:
%%bash
set -e
source /content/config.sh

mkdir -p "$ROOTDIR"

if [ -n "$VIDEOSOURCE" ]; then
  SRC_VIDEO="/content/drive/MyDrive/$VIDEOSOURCE"

  if [ ! -f "$SRC_VIDEO" ]; then
    echo "❌ Video not found: $SRC_VIDEO"
  else
    echo "🎬 Copying video: $SRC_VIDEO"
    cp -f "$SRC_VIDEO" "$ROOTDIR/video.mp4"
  fi
fi

if [ -n "$IMAGESET" ]; then
  SRC_IMAGES="/content/drive/MyDrive/$IMAGESET"
  DST_IMAGES="$ROOTDIR/images"

  if [ ! -d "$SRC_IMAGES" ]; then
    echo "❌ Image dataset not found: $SRC_IMAGES"
  else
    echo "🖼️ Preparing image dataset: $SRC_IMAGES"

    mkdir -p "$DST_IMAGES"

    COUNT=$(find "$SRC_IMAGES" -type f \( -iname "*.jpg" -o -iname "*.jpeg" -o -iname "*.png" \) | wc -l | tr -d ' ')
    if [ "$COUNT" -lt 2 ]; then
      echo "❌ Not enough images ($COUNT)"
      exit 1
    fi

    rm -f "$DST_IMAGES"/frame_*.png 2>/dev/null || true

    i=1
    for img in $(find "$SRC_IMAGES" -type f \( -iname "*.jpg" -o -iname "*.jpeg" -o -iname "*.png" \) | sort); do
      printf -v idx "%05d" "$i"
      cp "$img" "$DST_IMAGES/frame_${idx}.png"
      i=$((i+1))
    done

    echo "✅ Dataset ready in $DST_IMAGES"
  fi
fi

🎬 Copying video: /content/drive/MyDrive/gsplat/input/IMG_4797.MOV
❌ Image dataset not found: /content/drive/MyDrive/gsplat/input/perfume/video


In [ ]:
!git clone https://github.com/NicoIGN/video_to_ply.git
%cd video_to_ply

Cloning into 'video_to_ply'...
remote: Enumerating objects: 1170, done.
remote: Counting objects: 100% (177/177), done.
remote: Compressing objects: 100% (115/115), done.
remote: Total 1170 (delta 89), reused 136 (delta 52), pack-reused 993 (from 1)
Receiving objects: 100% (1170/1170), 754.92 KiB | 3.04 MiB/s, done.
Resolving deltas: 100% (739/739), done.
/content/video_to_ply


In [ ]:
%%bash
cd /content/video_to_ply
source /content/config.sh
git stash save && git checkout $GIT_BRANCH && git pull

No local changes to save
Your branch is up to date with 'origin/dev'.
Updating 679977e..20e3f93
Fast-forward
 scripts/preprocess_nerfstudio.sh | 80 +++++++++++++++++++++++++++-------------
 1 file changed, 54 insertions(+), 26 deletions(-)


Already on 'dev'
From https://github.com/NicoIGN/video_to_ply
   679977e..20e3f93  dev        -> origin/dev


In [ ]:
!RUN=0; \
[ "$RUN" -eq 0 ] && echo "skipping this stage" || \
( source /usr/local/miniforge/etc/profile.d/conda.sh && mamba env remove -y -n gsplat )

info     libmamba ****************** Backtrace Start ******************
debug    libmamba Loading configuration
trace    libmamba Compute configurable 'create_base'
trace    libmamba Compute configurable 'no_env'
trace    libmamba Compute configurable 'no_rc'
trace    libmamba Compute configurable 'rc_files'
trace    libmamba Compute configurable 'root_prefix'
trace    libmamba Get RC files configuration from locations up to HomeDir
trace    libmamba Configuration not found at '/root/.mambarc'
trace    libmamba Configuration not found at '/root/.mamba/mambarc.d'
trace    libmamba Configuration not found at '/root/.mamba/mambarc'
trace    libmamba Configuration not found at '/root/.mamba/.mambarc'
trace    libmamba Configuration not found at '/root/.config/mamba/mambarc.d'
trace    libmamba Configuration not found at '/root/.config/mamba/mambarc'
trace    libmamba Configuration not found at '/root/.config/mamba/.mambarc'
trace    libmamba Configuration not found at '/root/.condarc'
trac

In [ ]:
!source /usr/local/miniforge/etc/profile.d/conda.sh && \
mamba env list | grep -q gsplat && \
cd /content/video_to_ply/ && \
mamba run -n gsplat mamba env update -n gsplat -f environment/conda_colab.yml --prune -y || \
mamba env create -n gsplat -f environment/conda_colab.yml -y

[+] 0.0s
[+] 0.0s
[+] 0.1s
[+] 0.2s
conda-forge/linux-64  ⣾  
conda-forge/noarch    ⣾  
pytorch/linux-64      ⣾  
pytorch/noarch        ⣾  
nvidia/noarch         ⣾  [+] 0.3s
conda-forge/linux-64   3%
conda-forge/noarch     2%[+] 0.4s
conda-forge/linux-64   5%
conda-forge/noarch     5%[+] 0.5s
conda-forge/linux-64   6%
conda-forge/noarch    10%[+] 0.6s
conda-forge/linux-64   9%
conda-forge/noarch    16%[+] 0.7s
conda-forge/linux-64  12%
conda-forge/noarch    18%[+] 0.8s
conda-forge/linux-64  13%
conda-forge/noarch    23%[+] 0.9s
conda-forge/linux-64  15%
conda-forge/noarch    27%[+] 1.0s
conda-forge/linux-64  16%
conda-forge/noarch    29%[+] 1.1s
conda-forge/linux-64  19%
conda-forge/noarch    36%[+] 1.2s
conda-forge/linux-64  21%
conda-forge/noarch    38%[+] 1.3s
conda-forge/linux-64  25%
conda-forge/noarch    45%[+] 1.4s
conda-forge/linux-64  30%
conda-forge/noarch    55%[+] 1.5s
conda-forge/linux-64  32%
conda-forge/noarch    64%[+] 1.6s
conda-forge/linux-64  37%
conda-forge/noarch  

KeyboardInterrupt: 

In [ ]:
%%bash
source /usr/local/miniforge/etc/profile.d/conda.sh

SITE_PACKAGES=$(mamba run -n gsplat python -c "import site; print(site.getsitepackages()[0])")
TARGET="$SITE_PACKAGES/SuperGluePretrainedNetwork"

if [ ! -d "$TARGET" ]; then
    git clone --depth 1 \
        https://github.com/magicleap/SuperGluePretrainedNetwork.git \
        "$TARGET"
else
    echo "SuperGluePretrainedNetwork already installed."
fi

Cloning into '/usr/local/miniforge/envs/gsplat/lib/python3.10/site-packages/SuperGluePretrainedNetwork'...


In [ ]:
!source /usr/local/miniforge/etc/profile.d/conda.sh && \
source /content/config.sh && unset NUM_FRAMES && \
echo INPUT_MODE=$INPUT_MODE && \
cd /content/video_to_ply/ && \
INPUT_ARG="" && \
if [ "$INPUT_MODE" = "images" ] && [ -n "$IMAGESET" ]; then \
  INPUT_ARG="--images $ROOTDIR/images --name $BASENAME"; \
elif [ "$INPUT_MODE" = "video" ] && [ -n "$VIDEOSOURCE" ]; then \
  INPUT_ARG="--video $ROOTDIR/video.mp4 --fps $FPS --name $BASENAME"; \
fi && \
mamba run -n gsplat bash run.sh $INPUT_ARG --root "$ROOTDIR" --skip-conda --skip-filter --profile "$PROFILE" --no-proxy

Le flux de sortie a été tronqué et ne contient que les 5000 dernières lignes.
1930 (32.17%)       76.503 ms            5 m, 11 s            9.71 M                                     
1940 (32.33%)       78.406 ms            5 m, 18 s            9.46 M                                     
1950 (32.50%)       77.571 ms            5 m, 14 s            9.57 M                                     
Step (% Done)       Train Iter (time)    ETA (time)           Train Rays / Sec     Test Rays / Sec       
-------------------------------------------------------------------------------------------------------- 
1870 (31.17%)       67.875 ms            4 m, 40 s            10.96 M                                    
1880 (31.33%)       68.334 ms            4 m, 41 s            10.85 M                                    
1890 (31.50%)       68.282 ms            4 m, 40 s            10.86 M                                    
1900 (31.67%)       70.562 ms            4 m, 49 s            10.70 M     

In [ ]:
!RUN=0; \
[ "$RUN" -eq 0 ] && echo "skipping this stage" || \
( source /usr/local/miniforge/etc/profile.d/conda.sh && \
  source /content/config.sh && \
  cd /content/video_to_ply/ && \
  INPUT_ARG="" && \
  if [ "$INPUT_MODE" = "images" ] && [ -n "$IMAGESET" ]; then \
    INPUT_ARG="--images $ROOTDIR/images --name $BASENAME"; \
  elif [ "$INPUT_MODE" = "video" ] && [ -n "$VIDEOSOURCE" ]; then \
    INPUT_ARG="--video $ROOTDIR/video.mp4 --name $BASENAME"; \
  fi && \
  mamba run -n gsplat bash run.sh $INPUT_ARG \
    --root "$ROOTDIR" \
    --skip-conda \
    --profile "$PROFILE" \
    --no-proxy \
    --skip-conda \
    --skip-frame-extraction \
    --skip-colmap \
    --skip-training )

skipping this stage


In [ ]:
from google.colab import files
import os
import subprocess

# =========================
# LOAD CONFIG.SH VARIABLES
# =========================
result = subprocess.run(
    "source /content/config.sh && env",
    shell=True,
    executable="/bin/bash",
    capture_output=True,
    text=True,
)

for line in result.stdout.splitlines():
    if "=" in line:
        key, value = line.split("=", 1)
        os.environ[key] = value

# =========================
# CONFIG
# =========================
rootdir = os.environ.get("ROOTDIR", "/content/work")
export_dir = os.path.join(rootdir, "exports")
basename = os.environ.get("BASENAME", "")

if not basename:
    print("❌ BASENAME is not set")
    raise SystemExit(1)

base_ply = os.path.join(export_dir, f"{basename}.ply")

# =========================
# EXPORT ORIGINAL PLY
# =========================
if os.path.exists(base_ply):
    print(f"⬇️ Downloading original PLY: {os.path.basename(base_ply)}")
    files.download(base_ply)
else:
    print(f"⚠️ Original PLY not found: {base_ply}")

In [ ]:
from google.colab import files
import os
import subprocess
import glob

# =========================
# LOAD CONFIG.SH VARIABLES
# =========================
result = subprocess.run(
    "source /content/config.sh && env",
    shell=True,
    executable="/bin/bash",
    capture_output=True,
    text=True,
)

for line in result.stdout.splitlines():
    if "=" in line:
        key, value = line.split("=", 1)
        os.environ[key] = value

# =========================
# CONFIG
# =========================
rootdir = os.environ.get("ROOTDIR", "/content/work")
export_dir = os.path.join(rootdir, "exports")
basename = os.environ.get("BASENAME", "")

if not basename:
    print("❌ BASENAME is not set")
    raise SystemExit(1)

# =========================
# FIND FILTERED PLYS
# =========================
filtered_plys = sorted(
    glob.glob(os.path.join(export_dir, f"{basename}_*.ply"))
)

# =========================
# EXPORT FILTERED PLYS
# =========================
if not filtered_plys:
    print("⚠️ No filtered PLY files found.")
    print(f"📂 Searched: {export_dir}")
else:
    print(f"📦 Found {len(filtered_plys)} filtered PLY file(s)")

    for ply_path in filtered_plys:
        print(f"⬇️ Downloading filtered PLY: {os.path.basename(ply_path)}")
        files.download(ply_path)

⚠️ No filtered PLY files found.
📂 Searched: /content/exterieur/exports
